# GPU vs CPU Accuracy and Performance Comparison
This notebook compares the GPU-accelerated Poisson solver against the original CPU implementation. It focuses on the uniform grid case (`azu_unif=2`, `rad_unif=1`) as the nonuniform GPU components are not yet implemented.

We will compare:
1. **Accuracy**: The L2 relative error of the GPU solver vs. the CPU solver.
2. **Performance**: The speedup achieved by using the GPU.

In [7]:
# === 1. Install Dependencies ===
# This cell installs the required packages. 
# After running it, you MUST restart the kernel for the changes to take effect.
# In Google Colab: Runtime -> Restart runtime.

# Force uninstall any existing cupy versions to prevent conflicts
!pip uninstall -y -q cupy cupy-cuda11x cupy-cuda12x
!pip install -q finufft pynufft cupy-cuda12x pandas matplotlib

print("\nDependencies installed. Please RESTART the runtime now, then run the cells below.")


Dependencies installed. Please RESTART the runtime now, then run the cells below.


In [8]:
# === 2. Setup Environment and Imports ===
# Run this cell AFTER restarting the kernel.

import os
import sys
import pandas as pd
import cupy as cp

# Define project root and clone if not present
repo_root = "/content/NUFFTRR_Poisson"
repo_url = "https://github.com/CharliePyle4/NUFFTRR_Poisson.git"

if not os.path.exists(repo_root):
    print(f"Cloning {repo_url}...")
    !git clone {repo_url} {repo_root}
else:
    print("Pulling latest changes...")
    !git -C {repo_root} pull

# Set up path for imports
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Import project-specific helpers
from Tests.CPUvsGPU.helpers import (
    run_comparison_pipeline,
    render_accuracy_tables,
    render_performance_tables,
    _prepare_table2_df
)

# Check for GPU
try:
    print(f"CuPy is using GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
except Exception as e:
    print(f"CuPy GPU check failed: {e}")

Pulling latest changes...
Already up to date.


ImportError: cannot import name 'render_accuracy_tables' from 'Tests.CPUvsGPU.helpers' (/content/NUFFTRR_Poisson/Tests/CPUvsGPU/helpers.py)

# Parameters

In [ ]:
N_vals = [32, 64, 128, 256, 512]
M_vals = [32, 64, 128, 256, 512]
N_fixed = 32

MUTE_OUTPUT = False

# Run Comparison

In [ ]:
print("Running comparison for varying N and M values...")
df_nm = run_comparison_pipeline(N_vals, M_vals, test_type="VaryingNM", mute=MUTE_OUTPUT)

print("\nRunning comparison for varying M, BC, and Quadrature Rule (N fixed at 32)...")
df_bc_quad = run_comparison_pipeline(None, M_vals, test_type="VaryingM_BC_Quad", fixed_val=N_fixed, mute=MUTE_OUTPUT)

print("\nDone.")

Running comparison for varying N and M values...
  N=  32, M=  32 | Speedup=0.00x | Acc. Diff=4.213e-18
  N=  32, M=  64 | Speedup=0.11x | Acc. Diff=8.377e-19
  N=  32, M= 128 | Speedup=0.10x | Acc. Diff=8.100e-20
  N=  32, M= 256 | Speedup=0.10x | Acc. Diff=2.932e-18
  N=  32, M= 512 | Speedup=0.11x | Acc. Diff=3.153e-18
  N=  64, M=  32 | Speedup=0.03x | Acc. Diff=1.271e-19
  N=  64, M=  64 | Speedup=0.11x | Acc. Diff=9.868e-19
  N=  64, M= 128 | Speedup=0.14x | Acc. Diff=2.544e-18
  N=  64, M= 256 | Speedup=0.12x | Acc. Diff=4.547e-19
  N=  64, M= 512 | Speedup=0.12x | Acc. Diff=1.852e-18
  N= 128, M=  32 | Speedup=0.03x | Acc. Diff=6.133e-19
  N= 128, M=  64 | Speedup=0.15x | Acc. Diff=2.660e-18
  N= 128, M= 128 | Speedup=0.16x | Acc. Diff=2.938e-18
  N= 128, M= 256 | Speedup=0.16x | Acc. Diff=1.892e-18
  N= 128, M= 512 | Speedup=0.16x | Acc. Diff=1.017e-18
  N= 256, M=  32 | Speedup=0.05x | Acc. Diff=9.656e-19
  N= 256, M=  64 | Speedup=0.21x | Acc. Diff=2.370e-18
  N= 256, M= 128

/content/NUFFTRR_Poisson/Poisson_Solver/cpu_solver/radial/radial.py:92: RuntimeWarning: divide by zero encountered in scalar divide
  ratio = r_m[i] / r_m[i - 2]


  N=32, M=  32, Simp, Diri | Speedup=0.01x | Acc. Diff=2.541e-20
  N=32, M=  32, Simp, Neum | Speedup=0.02x | Acc. Diff=2.329e-18
  N=32, M=  64, Trap, Diri | Speedup=0.11x | Acc. Diff=8.377e-19
  N=32, M=  64, Trap, Neum | Speedup=0.11x | Acc. Diff=6.573e-19
  N=32, M=  64, Simp, Diri | Speedup=0.01x | Acc. Diff=1.470e-19
  N=32, M=  64, Simp, Neum | Speedup=0.01x | Acc. Diff=2.407e-19
  N=32, M= 128, Trap, Diri | Speedup=0.10x | Acc. Diff=8.100e-20
  N=32, M= 128, Trap, Neum | Speedup=0.11x | Acc. Diff=1.296e-17
  N=32, M= 128, Simp, Diri | Speedup=0.01x | Acc. Diff=3.136e-19
  N=32, M= 128, Simp, Neum | Speedup=0.01x | Acc. Diff=1.574e-19
  N=32, M= 256, Trap, Diri | Speedup=0.10x | Acc. Diff=2.932e-18
  N=32, M= 256, Trap, Neum | Speedup=0.10x | Acc. Diff=6.375e-17
  N=32, M= 256, Simp, Diri | Speedup=0.01x | Acc. Diff=2.063e-18
  N=32, M= 256, Simp, Neum | Speedup=0.01x | Acc. Diff=1.086e-17
  N=32, M= 512, Trap, Diri | Speedup=0.10x | Acc. Diff=3.153e-18
  N=32, M= 512, Trap, Neu

# Results: Varying N and M

In [ ]:
render_multitable(
    df_nm,
    index_col="N",
    columns_col="M",
    value_cols=["L2_rel_cpu", "L2_rel_gpu", "accuracy_diff", "runtime_cpu", "runtime_gpu", "speedup"],
    titles=["CPU Accuracy (L2 Rel. Error)", "GPU Accuracy (L2 Rel. Error)", "Accuracy Difference (Abs)", 
            "CPU Runtime (s)", "GPU Runtime (s)", "Speedup (CPU/GPU)"],
    float_formats=[
        lambda x: f"{x:.2e}", lambda x: f"{x:.2e}", lambda x: f"{x:.2e}",
        lambda x: f"{x:.4f}", lambda x: f"{x:.4f}", lambda x: f"{x:.2f}x"
    ]
)


 Accuracy and Timings 

--- CPU Accuracy (L2 Rel. Error) ---


M,32,64,128,256,512
N,,,,,
32,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
64,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
128,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
256,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
512,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08



--- GPU Accuracy (L2 Rel. Error) ---


M,32,64,128,256,512
N,,,,,
32,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
64,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
128,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
256,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08
512,1.10e-05,2.66e-06,6.55e-07,1.62e-07,4.04e-08



--- Accuracy Difference (Abs) ---


...

M,32,64,128,256,512
N,,,,,
32,0.0265,0.0018,0.0028,0.0050,0.0099
64,0.0013,0.0023,0.0038,0.0061,0.0137
128,0.0015,0.0024,0.0043,0.0079,0.0153
256,0.0020,0.0033,0.0060,0.0117,0.0274
512,0.0030,0.0053,0.0100,0.0257,0.0573



--- GPU Runtime (s) ---


M,32,64,128,256,512
N,,,,,
32,7.6483,0.0166,0.0275,0.0498,0.0937
64,0.0425,0.0216,0.0276,0.0519,0.1122
128,0.0441,0.0160,0.0272,0.0504,0.0955
256,0.0426,0.0160,0.0274,0.0515,0.0964
512,0.0448,0.0167,0.0282,0.0606,0.2324



--- Speedup (CPU/GPU) ---


M,32,64,128,256,512
N,,,,,
32,0.00x,0.11x,0.10x,0.10x,0.11x
64,0.03x,0.11x,0.14x,0.12x,0.12x
128,0.03x,0.15x,0.16x,0.16x,0.16x
256,0.05x,0.21x,0.22x,0.23x,0.28x
512,0.07x,0.32x,0.36x,0.42x,0.25x


# Results: Varying M, BC, and Quadrature

In [ ]:
df_bc_quad_fmt = _prepare_table2_df(df_bc_quad)
render_multitable(
    df_bc_quad_fmt,
    index_col="M",
    columns_col=["quad_str", "bc_str"],
    value_cols=["L2_rel_cpu", "L2_rel_gpu", "accuracy_diff", "runtime_cpu", "runtime_gpu", "speedup"],
    titles=["CPU Accuracy (L2 Rel. Error)", "GPU Accuracy (L2 Rel. Error)", "Accuracy Difference (Abs)", 
            "CPU Runtime (s)", "GPU Runtime (s)", "Speedup (CPU/GPU)"],
    float_formats=[
        lambda x: f"{x:.2e}", lambda x: f"{x:.2e}", lambda x: f"{x:.2e}",
        lambda x: f"{x:.4f}", lambda x: f"{x:.4f}", lambda x: f"{x:.2f}x"
    ]
)


 Accuracy and Timings 

--- CPU Accuracy (L2 Rel. Error) ---



--- GPU Accuracy (L2 Rel. Error) ---



--- Accuracy Difference (Abs) ---



--- CPU Runtime (s) ---



--- GPU Runtime (s) ---



--- Speedup (CPU/GPU) ---
